# Gaussianité de l'espace latent & Visualisation PLS — xAAEnet / Sleep Apnea
**Objectif** :
1. Tester la gaussianité de l'espace latent Ze via le test **Henze-Zirkler** (+ Shapiro-Wilk par dimension) — extrait de `extract_ze_and_visualize.ipynb`
2. Visualiser l'espace latent en 2D avec une **projection PLS supervisée** par le score de sévérité — extrait de `biomarkers_aae_latent.ipynb`

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE 1 — Installation des dépendances
# ═══════════════════════════════════════════════════════════════════
!pip install pingouin scikit-learn scipy numpy matplotlib seaborn -q


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE 2 — Imports globaux
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, HuberRegressor
from scipy import stats
import pingouin as pg
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng  = np.random.default_rng(SEED)

# ── Style global (dark theme) ────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.labelsize":     11,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "legend.fontsize":    9,
    "figure.facecolor":   "#0e1117",
    "axes.facecolor":     "#161b22",
    "axes.edgecolor":     "#30363d",
    "axes.labelcolor":    "#c9d1d9",
    "xtick.color":        "#8b949e",
    "ytick.color":        "#8b949e",
    "text.color":         "#c9d1d9",
    "grid.color":         "#21262d",
    "grid.linestyle":     "--",
    "grid.linewidth":     0.5,
    "legend.facecolor":   "#161b22",
    "legend.edgecolor":   "#30363d",
    "figure.dpi":         120,
})

print("✔ Imports OK")


## 1. Chargement de l'espace latent Ze

Charger le vecteur latent `Ze` depuis un fichier `.pt` (PyTorch) **ou** depuis un `.npy`.  
Adapter `PT_PATH` selon votre environnement.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE 3 — Chargement de Ze
#  • Option A : fichier .pt PyTorch (sans dépendance torch)
#  • Option B : torch.load() si torch est disponible
#  • Option C : np.load() pour un fichier .npy
# ═══════════════════════════════════════════════════════════════════
import zipfile, os

PT_PATH = "z_aae_final.pt"   # ← adapter le chemin ici

# ── Option A : lecture directe du ZIP (pas besoin de torch) ────────
if os.path.exists(PT_PATH):
    with zipfile.ZipFile(PT_PATH, "r") as z:
        # Le tensor brut est dans le premier fichier de données
        data_files = [n for n in z.namelist() if n.endswith("/data/0") or n == "data/0"]
        if data_files:
            raw = z.read(data_files[0])
            # Reshape : ajuster (N, D) selon votre espace latent
            N_TOTAL = len(raw) // (4 * 128)   # float32 = 4 bytes, D=128
            Ze_np = np.frombuffer(raw, dtype=np.float32).reshape(N_TOTAL, 128).copy()
        else:
            # Fallback : essayer torch si disponible
            import torch
            Ze_np = torch.load(PT_PATH, map_location='cpu').numpy()
else:
    # ── Option C : .npy ───────────────────────────────────────────────
    Ze_np = np.load(PT_PATH.replace('.pt', '.npy'))

N, D = Ze_np.shape
print(f"✔ Ze chargé : {Ze_np.shape}")
print(f"  mean={Ze_np.mean():.4f}  std={Ze_np.std():.4f}")
print(f"  min={Ze_np.min():.4f}  max={Ze_np.max():.4f}")
print(f"  Sparsité (zéros) : {(Ze_np == 0).mean():.1%}")


---
## 2. Test de gaussianité — Henze-Zirkler

> Extrait de `extract_ze_and_visualize.ipynb` (Cellule 9)

Le test **Henze-Zirkler** est un test multivarié de normalité basé sur la fonction caractéristique empirique.  
Il est appliqué :
- sur l'espace complet 128D (sous-échantillonné à N=500)
- sur une réduction PCA à 10 composantes

Le **test de Shapiro-Wilk** est appliqué dimension par dimension pour identifier les axes non-gaussiens.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE 4 — Test de gaussianité Henze-Zirkler + Shapiro-Wilk
#  Source : extract_ze_and_visualize.ipynb — Cellule 9
# ═══════════════════════════════════════════════════════════════════
Z_np = Ze_np
N_SAMPLE = 500
idx = rng.choice(len(Z_np), N_SAMPLE, replace=False)
Z_sample = Z_np[idx]

# ── Henze-Zirkler (128D) ────────────────────────────────────────────
hz, pval, normal = pg.multivariate_normality(Z_sample, alpha=0.05)
print(f"Henze-Zirkler (N={N_SAMPLE}, D=128)")
print(f"  HZ statistic : {hz:.4f}")
print(f"  p-value      : {pval:.6f}")
print(f"  Gaussien ?   : {'✔ OUI' if normal else '✘ NON'}")

# ── HZ sur PCA-10 ───────────────────────────────────────────────────
pca = PCA(n_components=10, random_state=SEED)
Z_pca = pca.fit_transform(StandardScaler().fit_transform(Z_np))
hz10, pval10, normal10 = pg.multivariate_normality(Z_pca[idx], alpha=0.05)
print(f"\nHenze-Zirkler sur PCA-10 (N={N_SAMPLE})")
print(f"  HZ statistic : {hz10:.4f}")
print(f"  p-value      : {pval10:.6f}")
print(f"  Gaussien ?   : {'✔ OUI' if normal10 else '✘ NON'}")

# ── Shapiro-Wilk par dimension ──────────────────────────────────────
pvals_sw = np.array([stats.shapiro(Z_np[idx, d])[1] for d in range(Z_np.shape[1])])
n_ok = (pvals_sw > 0.05).sum()
print(f"\nShapiro-Wilk par dimension")
print(f"  Dims gaussiennes (p>0.05) : {n_ok} / {Z_np.shape[1]}")
print(f"  p-value médiane           : {np.median(pvals_sw):.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE 5 — Figure : Distributions marginales + Shapiro-Wilk + QQ
#  Source : extract_ze_and_visualize.ipynb — Cellule 10
# ═══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0e1117')

# ── Distribution marginale (4 premières dims vs N(0,1)) ─────────────
ax = axes[0]
ax.set_facecolor('#161b22')
colors_dim = ['#63b3ed', '#56de91', '#e8a84a', '#fc6e6e']
for d, c in zip(range(4), colors_dim):
    ax.hist(Z_np[:, d], bins=60, alpha=0.5, density=True, color=c, label=f'dim {d}')
x_range = np.linspace(Z_np[:, :4].min(), Z_np[:, :4].max(), 300)
ax.plot(x_range, stats.norm.pdf(x_range, Z_np.mean(), Z_np.std()),
        'w--', lw=1.5, label='N(μ,σ)')
ax.set_title('Distributions marginales (dims 0–3)', color='white', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

# ── Shapiro-Wilk p-values par dimension ─────────────────────────────
ax = axes[1]
ax.set_facecolor('#161b22')
bar_colors = ['#56de91' if p > 0.05 else '#fc6e6e' for p in pvals_sw]
ax.bar(range(len(pvals_sw)), pvals_sw, color=bar_colors, width=1.0)
ax.axhline(0.05, color='white', lw=1.2, ls='--', label='α = 0.05')
ax.set_title(f'Shapiro-Wilk par dimension\n{n_ok}/{Z_np.shape[1]} dims gaussiennes',
             color='white', fontweight='bold')
ax.set_xlabel('Dimension', fontsize=10)
ax.set_ylabel('p-value', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.12)

# ── Q-Q plot dim 0 ──────────────────────────────────────────────────
ax = axes[2]
ax.set_facecolor('#161b22')
(osm, osr), (slope, intercept, r) = stats.probplot(Z_np[:, 0])
ax.scatter(osm, osr, s=1.5, alpha=0.3, color='#63b3ed')
ax.plot(osm, slope * np.array(osm) + intercept, 'w--', lw=1.5, label=f'Droite théorique (r={r:.3f})')
ax.set_title('Q-Q plot — dim 0', color='white', fontweight='bold')
ax.set_xlabel('Quantiles théoriques N(0,1)', fontsize=10)
ax.set_ylabel('Quantiles observés', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.12)

# ── Résumé HZ en titre ──────────────────────────────────────────────
hz_verdict  = '✔ Gaussien' if normal  else '✘ Non-gaussien'
hz10_verdict = '✔ Gaussien' if normal10 else '✘ Non-gaussien'
fig.suptitle(
    f'Gaussianité de Ze  |  HZ-128D : {hz_verdict} (p={pval:.4f})  '
    f'·  HZ-PCA10 : {hz10_verdict} (p={pval10:.4f})',
    fontsize=11, fontweight='bold', color='#c9d1d9', y=1.02
)
plt.tight_layout()
fig.savefig('fig_gaussianity.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print('✔ Sauvegardé : fig_gaussianity.png')
plt.show()


---
## 3. Visualisation PLS de l'espace latent

> Extrait de `biomarkers_aae_latent.ipynb` (Cellules 3 & 4)

La **Partial Least Squares Regression (PLS)** projette Ze en 2D en **maximisant la covariance** avec le score de sévérité.  
Les axes PLS sont donc supervisés : ils révèlent la direction de séparation clinique dans l'espace latent.

> ⚠️ Si vous n'avez pas de labels de sévérité réels, un proxy est généré via PCA + sigmoïde (voir cellule ci-dessous).


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE 6 — Score de sévérité
#  Remplacer severity_vec par vos vrais labels si disponibles.
#  Sinon : proxy via PCA 5D + sigmoïde (même logique que biomarkers nb)
# ═══════════════════════════════════════════════════════════════════
# ── Option A : vos vrais labels ─────────────────────────────────────
# severity_vec = np.load('severity_labels.npy').astype(np.float32)

# ── Option B : proxy PCA → sigmoïde ─────────────────────────────────
_pca_sev = PCA(n_components=5, random_state=SEED)
_scores  = _pca_sev.fit_transform(Ze_np)
sev_weights = np.array([0.6, 0.4, -0.3, 0.2, 0.1])
raw_sev = _scores @ sev_weights
noise   = rng.standard_normal(N) * 0.05
severity_vec = (1.0 / (1.0 + np.exp(-(raw_sev + noise)))).astype(np.float32)

N_TRAIN = len(Ze_np)
Z_train = Ze_np

print(f"✔ severity_vec : min={severity_vec.min():.3f}  max={severity_vec.max():.3f}  mean={severity_vec.mean():.3f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE 7 — Régression de la direction de sévérité dans Ze (128D)
#  Source : biomarkers_aae_latent.ipynb — Cellule 3
# ═══════════════════════════════════════════════════════════════════
print("▶  Regressing severity direction in latent space …")

reg_sev = HuberRegressor(max_iter=500)
reg_sev.fit(Z_train, severity_vec)
severity_direction_128 = reg_sev.coef_    # vecteur 128D

print(f"Severity direction norm : {np.linalg.norm(severity_direction_128):.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE 8 — Figure : Espace latent supervisé PLS (2D)
#  Source : biomarkers_aae_latent.ipynb — Cellule 4
# ═══════════════════════════════════════════════════════════════════
print("▶  Fitting PLS and plotting …")

# ── 1. PLS fit ──────────────────────────────────────────────────────
scaler_pls = StandardScaler()
Z_train_sc = scaler_pls.fit_transform(Z_train)

pls = PLSRegression(n_components=2, scale=False, max_iter=1000)
pls.fit(Z_train_sc, severity_vec)
T_train = pls.transform(Z_train_sc)          # (N, 2)

# ── 2. Pseudo-R² de chaque composante PLS vs sévérité ───────────────
r2_c1 = stats.pearsonr(T_train[:, 0], severity_vec)[0] ** 2
r2_c2 = stats.pearsonr(T_train[:, 1], severity_vec)[0] ** 2

# ── 3. Champ de sévérité (Ridge sur T_train) ────────────────────────
reg2d = Ridge(alpha=1e-3)
reg2d.fit(T_train, severity_vec)

t1_min, t1_max = T_train[:, 0].min() - 0.5, T_train[:, 0].max() + 0.5
t2_min, t2_max = T_train[:, 1].min() - 0.5, T_train[:, 1].max() + 0.5
t1_grid, t2_grid = np.meshgrid(
    np.linspace(t1_min, t1_max, 300),
    np.linspace(t2_min, t2_max, 300),
)
grid_pts = np.c_[t1_grid.ravel(), t2_grid.ravel()]
sev_grid = reg2d.predict(grid_pts).reshape(t1_grid.shape)
sev_grid = np.clip(sev_grid, 0, 1)

sev_arrow = reg2d.coef_ / (np.linalg.norm(reg2d.coef_) + 1e-8)

# ── 4. Plot ─────────────────────────────────────────────────────────
fig1 = plt.figure(figsize=(14, 10))
fig1.patch.set_facecolor("#0e1117")

ax_main = fig1.add_subplot(111)
fig1.subplots_adjust(left=0.06, right=0.97, top=0.88, bottom=0.12)
ax_main.set_facecolor("#0d1117")

# Background gradient
ax_main.imshow(
    sev_grid,
    extent=[t1_min, t1_max, t2_min, t2_max],
    origin="lower", aspect="auto",
    cmap="magma", alpha=0.28, vmin=0, vmax=1, zorder=0,
)

# Iso-contours
cs = ax_main.contour(
    t1_grid, t2_grid, sev_grid,
    levels=np.linspace(0.1, 0.9, 9),
    cmap="magma", alpha=0.35, linewidths=0.7, zorder=1,
)
ax_main.clabel(cs, fmt="%.1f", fontsize=7, colors="#8b949e", inline=True)

# Scatter (coloré par sévérité)
sc_tr = ax_main.scatter(
    T_train[:, 0], T_train[:, 1],
    c=severity_vec, cmap="YlOrBr",
    s=40, alpha=0.75, linewidths=0.5, rasterized=False,
    norm=Normalize(0, 1), zorder=2,
    label=f"Samples (n={N_TRAIN})",
)

# Flèche gradient de sévérité
cx = T_train[:, 0].mean()
cy = T_train[:, 1].mean()
span = max(t1_max - t1_min, t2_max - t2_min) * 0.18
ax_main.annotate(
    "",
    xy=(cx + span * sev_arrow[0], cy + span * sev_arrow[1]),
    xytext=(cx, cy),
    arrowprops=dict(arrowstyle="-|>", color="#ff4444", lw=4.8, mutation_scale=20),
    zorder=6,
)
ax_main.text(
    cx + span * sev_arrow[0] * 1.42,
    cy + span * sev_arrow[1] * 1.12,
    "Severity\ngradient",
    color="#ff4444", fontsize=14, ha="center", va="center",
    fontweight="bold", zorder=6,
)

# Colorbar
cbar_tr = fig1.colorbar(
    ScalarMappable(norm=Normalize(0, 1), cmap="YlOrBr"),
    ax=ax_main, fraction=0.024, pad=0.01, shrink=0.85,
)
cbar_tr.set_label("Severity score", color="#e8c47e", labelpad=5)
plt.setp(cbar_tr.ax.yaxis.get_ticklabels(), color="#e8c47e")

# Labels & titre
ax_main.set_xlabel(f"PLS Component 1  (r² w/ severity = {r2_c1:.3f})", fontsize=10)
ax_main.set_ylabel(f"PLS Component 2  (r² w/ severity = {r2_c2:.3f})", fontsize=10)
ax_main.set_title(
    "Supervised PLS Projection  ·  128 → 2 dimensions\n"
    "Axes maximise covariance with severity score  —  background = predicted severity field",
    pad=10, loc="left", fontsize=11,
)

leg_el = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#d4a850",
           markersize=7, label=f"Sample (n={N_TRAIN})", linestyle="None"),
    Line2D([0], [0], color="#ff4444", lw=2, label="Regressed severity direction"),
    mpatches.Patch(facecolor="#7a3080", alpha=0.4, label="Severity iso-contours"),
]
ax_main.legend(handles=leg_el, loc="upper left", framealpha=0.3, borderpad=0.8)
ax_main.grid(True, alpha=0.10)

plt.suptitle(
    "Latent Space — Severity-Supervised Projection (PLS)",
    fontsize=13, fontweight="bold", y=0.97, color="#c9d1d9",
)

fig1.savefig("fig_pls_latent_space.png", dpi=150, bbox_inches="tight",
             facecolor=fig1.get_facecolor())
print("   ✔  Saved fig_pls_latent_space.png")
plt.show()


---
## Résumé des résultats

| Test | Statistique | p-value | Verdict |
|---|---|---|---|
| Henze-Zirkler (128D) | `hz` | `pval` | ✔ / ✘ |
| Henze-Zirkler (PCA-10) | `hz10` | `pval10` | ✔ / ✘ |
| Shapiro-Wilk (médiane) | — | `median(pvals_sw)` | n_ok/128 dims OK |

**PLS** : les deux composantes capturent la direction de sévérité principale dans l'espace latent Ze.
